#**ML Models (CPU)**

## Importing in Data

In [2]:
#!rm -r Multicore_2026S


In [3]:
!git clone https://github.com/seernono2001/Multicore_2026S

fatal: destination path 'Multicore_2026S' already exists and is not an empty directory.


In [4]:
#Make all datasets
import pandas
#results/barrier_heavy direcotry
barrier_heavy = pandas.read_csv('Multicore_2026S/results/results_barrier_heavy.csv')
compute = pandas.read_csv('Multicore_2026S/results/results_compute.csv')
stream = pandas.read_csv('Multicore_2026S/results/results_stream.csv')

#concatenate all data
data = pandas.concat([barrier_heavy, compute, stream])
print(data)


     Index  ProblemSize  Threads  Operation  AvgSqTime  AvgParTime  AvgSpeedup
0        1         1004       22          2   0.000947    0.004295    0.220492
1        2         1019       11          2   0.000969    0.002159    0.449193
2        3         1027       40          2   0.000965    0.007539    0.128020
3        4         1032       45          2   0.000973    0.009406    0.103844
4        5         1037       60          2   0.000981    0.042648    0.025595
..     ...          ...      ...        ...        ...         ...         ...
995    996     88728965       46          1   5.901622    0.693329    8.286426
996    997     91423191       46          1   1.270303    0.516925    2.519583
997    998     92325702       45          1   1.719941    0.333705    5.079473
998    999     95281654       52          1   1.338509    0.378780    3.534867
999   1000     97283652       15          1   2.296574    0.960196    3.117948

[3000 rows x 7 columns]


In [5]:
import pandas as pd

# Drop columns not used for modelling
# New CPU data columns: Index, ProblemSize, Threads, Operation, AvgSqTime, AvgParTime, AvgSpeedup
data_use = data.copy()
data_use = data_use.drop(['Index', 'AvgSqTime', 'AvgParTime'], axis=1)
print(data_use)


     ProblemSize  Threads  Operation  AvgSpeedup
0           1004       22          2    0.220492
1           1019       11          2    0.449193
2           1027       40          2    0.128020
3           1032       45          2    0.103844
4           1037       60          2    0.025595
..           ...      ...        ...         ...
995     88728965       46          1    8.286426
996     91423191       46          1    2.519583
997     92325702       45          1    5.079473
998     95281654       52          1    3.534867
999     97283652       15          1    3.117948

[3000 rows x 4 columns]


## Barrier Heavy Models

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
barrier_heavy = data_use[data_use['Operation'] == 2]

#scale and split data, drop speedup and GPU
X = barrier_heavy.drop(['AvgSpeedup','Operation'], axis=1)
y = barrier_heavy['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)




### Linear Regression Baseline

In [7]:
#import linear regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

#make model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#print deature importances along with their names
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])



MSE: 20.695184771532016
R2: 0.013925575223098563
ProblemSize : 0.6122132109292788
Threads : 0.5474391060258891


### Linear Regression (Big Problem Sizes)

In [8]:
#same thing but with problem sizes >1 mil
barrier_heavy_big = barrier_heavy[barrier_heavy['ProblemSize'] > 1000000]
print(barrier_heavy_big)
X = barrier_heavy_big.drop(['AvgSpeedup','Operation'], axis=1)
y = barrier_heavy_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4
                                                    )
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


     ProblemSize  Threads  Operation  AvgSpeedup
589      1001754       51          2   16.742466
590      1002543       17          2   10.846576
591      1002780       48          2   16.569466
592      1011433        3          2    3.022394
593      1020319       12          2    9.872406
..           ...      ...        ...         ...
995     88728965       46          2    6.474242
996     91423191       46          2   14.206313
997     92325702       45          2   11.504578
998     95281654       52          2    5.905959
999     97283652       15          2    4.432059

[411 rows x 4 columns]


In [9]:
#train
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 19.985776427526435
R2: 0.018724276122595085
ProblemSize : -0.16326160046536045
Threads : 1.033874116164945


### Random Forests (Baseline)

In [10]:
#use the same barrier_heavy
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score


#bring back barrier_heavy
barrier_heavy = data_use[data_use['Operation'] == 2]
X = barrier_heavy.drop(['AvgSpeedup','Operation'], axis=1)
y = barrier_heavy['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [11]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 5.674057462715894
R2: 0.729645180245256
ProblemSize : 0.7766211027922022
Threads : 0.2233788972077978


### Random Forests (Big Problem Sizes)

In [12]:
#bring back barrier_heavy_big
barrier_heavy_big = barrier_heavy[barrier_heavy['ProblemSize'] > 1000000]
X = barrier_heavy_big.drop(['AvgSpeedup','Operation'], axis=1)
y = barrier_heavy_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
#train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 9.51820768170612
R2: 0.5694163814925451
ProblemSize : 0.6850511426935068
Threads : 0.31494885730649325


## Stream Models

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
stream = data_use[data_use['Operation'] == 1]

#scale and split data, drop speedup and GPU
X = stream.drop(['AvgSpeedup','Operation'], axis=1)
y = stream['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



### Linear Regression Baseline

In [15]:
#same as barrier_heavy
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])


MSE: 2.3660135621586567
R2: 0.4091391794708068
ProblemSize : 1.2309588262847535
Threads : -0.18139358104937237


### Linear Regression (Big Problem Sizes)

In [16]:

stream_big = stream[stream['ProblemSize'] > 1000000]
X = stream_big.drop(['AvgSpeedup','Operation'], axis=1)
y = stream_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [17]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances same as earlier
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])



MSE: 1.1601105143344879
R2: 0.32345489568597874
ProblemSize : 0.8145561613203719
Threads : 0.033547394352897185


### Random Forests

In [18]:
#bring back sub data
stream = data_use[data_use['Operation'] == 1]
#print stream
print(stream)
#print barrier_heavya
print(barrier_heavy)
#save sub data as csv in direcotry
stream.to_csv('stream.csv', index=False)
# save add data
barrier_heavy.to_csv('barrier_heavy.csv', index=False)
X = stream.drop(['AvgSpeedup','Operation'], axis=1)
y = stream['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



     ProblemSize  Threads  Operation  AvgSpeedup
0           1004       22          1    0.005156
1           1019       11          1    0.010948
2           1027       40          1    0.002739
3           1032       45          1    0.002640
4           1037       60          1    0.001839
..           ...      ...        ...         ...
995     88728965       46          1    8.286426
996     91423191       46          1    2.519583
997     92325702       45          1    5.079473
998     95281654       52          1    3.534867
999     97283652       15          1    3.117948

[1000 rows x 4 columns]
     ProblemSize  Threads  Operation  AvgSpeedup
0           1004       22          2    0.220492
1           1019       11          2    0.449193
2           1027       40          2    0.128020
3           1032       45          2    0.103844
4           1037       60          2    0.025595
..           ...      ...        ...         ...
995     88728965       46          2    6.47

In [19]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.6976767422955966
R2: 0.8257702918487042
ProblemSize : 0.8751259469253356
Threads : 0.1248740530746644


### Random Forests (Big Problem Sizes)

In [20]:
#bring bad stream
stream_big = stream[stream['ProblemSize'] > 1000000]
X = stream_big.drop(['AvgSpeedup','Operation'], axis=1)
y = stream_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [21]:
#train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.8513621267685959
R2: 0.5035086125444801
ProblemSize : 0.7287810947835643
Threads : 0.27121890521643555


## Compute Models

In [22]:
#same pre processing as stream
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
compute = data_use[data_use['Operation'] == 0]
#scale and split data, drop gpu speedup etc
X = compute.drop(['AvgSpeedup','Operation'], axis=1)
y = compute['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)





### Linear Regression Baseline

In [23]:
#make model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.8975151420384967
R2: 0.2170765867598241
ProblemSize : 0.465868465827507
Threads : -0.26360610014219754


### Linear Regression (Big Problem Sizes)



In [24]:
#make the big problem set
compute_big = compute[compute['ProblemSize'] > 1000]
X = compute_big.drop(['AvgSpeedup','Operation'], axis=1)
y = compute_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [25]:
#make model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.8975151420384967
R2: 0.2170765867598241
ProblemSize : 0.465868465827507
Threads : -0.26360610014219754


### Linear Regression (Bigger Problem Sizes)


In [26]:
#now do when problem size >1500
compute_big = compute[compute['ProblemSize'] > 1500]
X = compute_big.drop(['AvgSpeedup','Operation'], axis=1)
y = compute_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [27]:
#make model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.9615359093354792
R2: 0.16723637486221765
ProblemSize : 0.46946279519702183
Threads : -0.2769657594601231


### Random Forests

In [28]:
#Bring back mult
compute = data_use[data_use['Operation'] == 0]
X = compute.drop(['AvgSpeedup','Operation'], axis=1)
y = compute['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [29]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])


MSE: 0.03110751871547793
R2: 0.9728641851380486
ProblemSize : 0.8705963650992434
Threads : 0.1294036349007566


### Random Forests (Big Problem Sizes)

---



In [30]:
#make bigger
compute_big = compute[compute['ProblemSize'] > 1000]
X = compute_big.drop(['AvgSpeedup','Operation'], axis=1)
y = compute_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [31]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.03110751871547793
R2: 0.9728641851380486
ProblemSize : 0.8705963650992434
Threads : 0.1294036349007566


### Random Forests (Bigger Problem Sizes)

In [32]:
#bigger problem size
compute_big = compute[compute['ProblemSize'] > 1500]
X = compute_big.drop(['AvgSpeedup','Operation'], axis=1)
y = compute_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [33]:
#make model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 0.05090799836680548
R2: 0.9559097805325356
ProblemSize : 0.867441202999688
Threads : 0.13255879700031203


## Everything Model

### Linear Regression


In [34]:
#use all data
#one hot encode the operation
data_use = pd.get_dummies(data_use, columns=['Operation'])
#scale and split data
X = data_use.drop(['AvgSpeedup'], axis=1)
y = data_use['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [35]:
#build model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 7.6430256710307365
R2: 0.3000665982709969
ProblemSize : 0.8015960125402746
Threads : -0.00953024693869775
Operation_0 : -0.7170276277184217
Operation_1 : -0.41749286666122115
Operation_2 : 1.1431998235745626


### Linear Regression (Big Problem Sizes)


In [36]:
#extract mult rows in data_use
compute = data_use[data_use['Operation_0'] == 1]
#cut rows with problem size less than 1000
compute_big = compute[compute['ProblemSize'] > 1000]
#extract the rest
ro_data = data_use[data_use['Operation_0'] == 0]
#cut rows with problem size less than 1 mil
ro_data_big = ro_data[ro_data['ProblemSize'] > 1000000]

data_use_big = pd.concat([compute_big, ro_data_big])
#same pre processing
X = data_use_big.drop(['AvgSpeedup'], axis=1)
y = data_use_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)




In [37]:
#model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

feature_names = X.columns
feature_importances = model.coef_
for i in range(len(feature_names)):
    print(feature_names[i], ':', feature_importances[i])

MSE: 4.66857798923464
R2: 0.4334700306778464
ProblemSize : 0.4336771377269153
Threads : 0.12231685968853223
Operation_0 : -1.0855291263959757
Operation_1 : -0.019317906458020007
Operation_2 : 1.3111606695076878


### Random Forests


In [38]:
#bring back all data
X = data_use.drop(['AvgSpeedup'], axis=1)
y = data_use['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [39]:
#build model
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
 print(feature_names[i], ':', feature_importances[i])

MSE: 1.3654148244548305
R2: 0.8749579703134774
ProblemSize : 0.5644229935326419
Threads : 0.1581844692703434
Operation_0 : 0.005675714056908355
Operation_1 : 0.006952618554322892
Operation_2 : 0.2647642045857834


### Random Forests (Bigger Problem Size)

In [40]:
#bring back
X = data_use_big.drop(['AvgSpeedup'], axis=1)
y = data_use_big['AvgSpeedup']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [41]:
#build model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)
#importances
feature_names = X.columns
feature_importances = model.feature_importances_
for i in range(len(feature_names)):
 print(feature_names[i], ':', feature_importances[i])

MSE: 2.642171020265516
R2: 0.6793736614218367
ProblemSize : 0.42984713818780734
Threads : 0.1759965244289372
Operation_0 : 0.04922759006165356
Operation_1 : 0.017064806489920356
Operation_2 : 0.32786394083168147
